# Introduction — An Almost Uniform Universe, to Structure

Although the early Universe was remarkably homogeneous, tiny differences in density existed: some regions contained slightly more matter than others.

A slightly overdense region has a slightly stronger gravitational attraction. It therefore attracts more matter. As more matter accumulates, the region becomes even denser, strengthening its gravitational attraction further.

Galaxies are not distributed randomly throughout space. Instead, observations reveal an enormous network of structure:

- galaxies gather into groups and clusters

- clusters are connected by long filaments

- broad sheets of matter surround large empty regions

- these low density regions are called cosmic voids

Thus, early universe contained tiny fluctuations in density.

The fundamental question we investigate here is:

> **How can tiny differences in the initial distribution of matter grow into large scale gravitational structure?**

The basic physical mechanism is:


$$
\boxed{
\text{Density fluctuation}
\rightarrow
\text{gravitational attraction}
\rightarrow
\text{matter accumulation}
\rightarrow
\text{larger density fluctuation}
}
$$


This is the idea of **gravitational instability**.

To investigate it computationally, we construct an **N-body simulation**.

### What Is an N-body Simulation?

An N-body system is a collection of \(N\) objects that interact with one another.

For example,

$$
N=2
$$

gives a two-body problem.

But if

$$
N=150,
$$

then every particle can gravitationally influence every other particle.

Our simulation therefore asks:

> **If 150 particles are initially distributed in a roughly concentrated cloud and given small induvisual random velocities, what happens when we allow them to interact gravitationally?**

We track two parameters of each of those particles

$$
\boxed{\text{position}}
$$

and

$$
\boxed{\text{velocity}}
$$

as the system evolves.

### What Do Our "Particles" Represent?

An important clarification is necessary here.

A particle in this simulation is **not a fundamental dark matter particle**.

Instead, it is a **simulation particle**: a coarse grained representative parcel of dark matter.

A real cosmological simulation cannot possibly follow every microscopic particle in the Universe.

Therefore, we coarse-grain the matter distribution:

$$
\text{Real Universe}
\rightarrow
\text{enormous number of microscopic particles}
\rightarrow
\text{coarse-grained representation}
\rightarrow
\text{N-body simulation particles}
$$

For this project, we interpret the particles as representing **collisionless dark matter**.

### Important distinction

Dark matter and dark energy are not the same thing.

Our particles represent matter.

Dark energy is **not explicitly represented** in this simulation.

# Importing necessary Libraries

We need two main Python libraries:

- **NumPy** for numerical calculations and arrays.
- **Matplotlib** for visualization.

NumPy will allow us to perform large numbers of calculations simultaneously using arrays.

Matplotlib will allow us to visualize how the particle distribution changes with time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Defining the Simulation Parameters

We now define the basic parameters of our numerical experiment.

We need to specify:

- how many particles we have,
- the gravitational constant in our simulation units,
- the size of each timestep,
- and how many timesteps we want to evolve.

In [ ]:
N_particles = 150

G = 1.0

dt = 0.01

num_steps = 100


**Number Of Particles**:

This means that our simulation contains

N=150

particles.

Each particle has a three-dimensional position,

(x,y,z),

and a three-dimensional velocity,

(vx ,vy, vz)

Therefore, each particle requires six numbers to describe its instantaneous state.

**Gravitational Constant**

$$ G = 1 $$

The real Newtonian gravitational constant is

$$ G≈6.674×10−11 m3kg−1s−2 $$

However, we are not working in SI units.

We use a custom system of simulation units and set

G=1.

This makes the numerical experiment easier to handle.

Therefore, the distances, velocities, masses and times in this simulation should be understood as simulation units, not literal metres, seconds and kilograms.

**Time Step**
$$ dt = 0.01 $$

computationally, we do not evolve the system continuously.

Instead, we divide time into small intervals:

Δt=0.01.

At each timestep we calculate the gravitational acceleration and use it to update the velocity and position.

A smaller timestep generally gives a more accurate approximation, although it requires more computation.

**Number of steps**

the simulation performs 100 updates. 
The total simulated time is therefore approximately

$$
T=Nsteps
	​

Δt
$$

so that

$$ T=100(0.01)=1 $$

simulation time unit.

# Making the conditions Reproducible

The random initial positions and velocities should be reproducable and should go on updating

Thus, we generate pseudo random numbers. By specifying a seed (ie. a starting number used to initialize a pseudo-random number generator), we ensure that the same sequence of random numbers is generated every time the code is run.

In [ ]:
np.random.seed(101)

The value `101` itself has no physical significance.

It simply fixes the random sequence for states of velocity and position


In [ ]:
positions = np.random.normal(0,2,(N_particles, 3))  # Shape: (N, 3) for X, Y, Z
velocities = np.random.normal(0, 0.5, (N_particles, 3)) # Small random initial velocity


# Softening

Newtonian gravity contains an inverse-square dependence:

$$
F\propto\frac{1}{r^2}.
$$

In vector form, the acceleration contains

$$
\frac{\mathbf r}{r^3}.
$$

If two particles become extremely close,

$$
r\rightarrow0,
$$

the gravitational acceleration approaches towards infinity.

At exactly

$$
r=0,
$$

the expression mimics a singularity.

This can cause numerical instability in an N-body calculation.

To avoid such infinite inconsistensies, we introduce a small softening length. 

$$
\epsilon=0.1.
$$

In [ ]:
softening = 0.1


Instead of using

$$
r^2,
$$

we use

$$
r^2+\epsilon^2.
$$

Thus the force remains finite even when two simulation particles become extremely close.

This is a numerical technique commonly used in computational simulations.

It also reflects the fact that a simulation particle is a coarse-grained representation of matter rather than a literal point particle.

# Computing gravitational Accelerations
### (The heart of the simulation)

Newton's law says every particle attracts every other particle.

- Particle 1 pulls

 Particle 2

 Particle 3

...

 Particle N

---

- Particle 2 pulls

 Particle 1

 Particle 3

...

 Particle N


Every particle interacts with every other particle.

This is called the N-body problem.


In [ ]:
def compute_gravitational_accelerations(pos):

    """
    Computes acceleration for all particles using vectorized numpy operations.
    Avoids slow python loops by computing distances via matrix broadcasting.
    """

    # Matrix slicing/broadcasting trick to get 3D separation matrices
    # x_ij = x_j - x_i

    # to compute a matrix of relative distances

    dx = pos[:, 0:1].T - pos[:, 0:1]  

    dy = pos[:, 1:2].T - pos[:, 1:2]

    dz = pos[:, 2:3].T - pos[:, 2:3]

NumPy allows us to replace these repeated calculations with **vectorized array operations**.

This is the central idea of the simulation.

We want to calculate the displacement between every pair of particles.

For the x-coordinate:

$$
dx_{ij}=x_j-x_i.
$$

(similar procedure for Y and Z components).

This means that instead of asking:

> "What is the x-separation between particle 17 and particle 93?"

we construct an entire matrix containing the x-separation of **every particle from every other particle simultaneously**.

### computational nuance:
- pos[: , 0:1] extracts a column vector of all \(X\)-coordinates (Shape: 150 * 1).

- ".T" transposes it into a row vector (Shape:1 * 150).

- When you subtract a column vector from a row vector, NumPy triggers broadcasting. It expands both vectors into matching (150 * 150) matrices and subtracts them.




# Distance

From three-dimensional geometry,

$$
r_{ij}^2
=
dx_{ij}^2+
dy_{ij}^2+
dz_{ij}^2.
$$

We can therefore calculate the squared distance between every pair simultaneously.


In [ ]:
r_squared = dx**2 + dy**2 + dz**2

Constructing the \(1/r^3\) Factor

The gravitational acceleration contains the factor

$$
\frac{1}{r^3}.
$$

With softening included, we use

$$
\frac{1}
{\left(r^2+\epsilon^2\right)^{3/2}}.
$$


In [ ]:
inv_r3 = (dx**2 + dy**2 + dz**2 + softening**2)**(-1.5)

### The Inverse-Cube Law ($r^{-3}$)

Newtonian gravitational acceleration follows the familiar inverse-square law:

$$a \propto \frac{1}{r^2}$$

However, because we are splitting this total acceleration into independent 3D dimensional components ($a_x, a_y, a_z$), we cannot just use the absolute distance. We must scale the total acceleration by the directional unit vector component ($\frac{dx}{r}$) to project it correctly onto that axis. 

Mathematically, for the X-axis:

$$a_x \propto \frac{1}{r^2} \times \frac{dx}{r} = \frac{dx}{r^3} = dx \cdot r^{-3}$$

To calculate this efficiently, we use a fractional exponent method. The Pythagorean theorem gives us the square of the distance:

$$r^2 = dx^2 + dy^2 + dz^2$$

By raising this base value ($r^2$) to the power of **$-1.5$**, we perfectly isolate our target inverse-cube value in a single computational step:

$$(r^2)^{-1.5} = r^{-3} = \frac{1}{r^3}$$


In [ ]:
    ax = G * (dx * inv_r3) @ np.ones(N_particles)

    ay = G * (dy * inv_r3) @ np.ones(N_particles)

    az = G * (dz * inv_r3) @ np.ones(N_particles)

     return np.vstack((ax, ay, az)).T


This chunk of code collapses the matrix of individual pairings back into a single vector of final forces.
- dx * inv_r3 performs element-wise multiplication to evaluate the term $$ \frac{dx_{ij}}{r_{ij}^{3}} $$ for every single pairing
- The @ operator stands for Matrix Multiplication

- Multiplying a 150 * 150 matrix by a 150 * 1 vector of np.ones(N_particles) sums up each row of the matrix

- "return np.vstack((ax, ay, az)).T" stacks the three flat vectors of length 150 vertically on top of each other. This creates a matrix of shape 3 * 150

- ".T" transposes (flips) it to a shape of 150 * 3. It allows us to directly update states later using simple array addition (positions += velocities * dt)

# The Time Loop (Running the Universe)

Now we must let time move forward. We do this using a partial **Leapfrog / Kick-Drift Integrator**. 

For every step in time:
1. We check where all particles currently sit.
2. We calculate gravity's pull at those exact coordinates (`compute_gravitational_accelerations`).
3. **The Kick**: We update the `velocities` based on the gravitational pull multiplied by our tiny time step (`dt`).
4. **The Drift**: We update the `positions` by moving the particles forward based on their newly calculated velocities.

### Storing History
Because we want to see the evolution later, we don't want to just look at the final result. We use the modulo operator (`step % 20 == 0`) to save a complete duplicate copy (`.copy()`) of our universe's coordinates every 20 steps. 


In [ ]:
# Create an empty list to act as our cosmic time capsule
history = [] 

# Loop 100 times to simulate 100 increments of time passing
for step in range(num_steps):
    
    # 1. Find out how hard everything is pulling right now
    acc = compute_gravitational_accelerations(positions)
    
    # 2. Update velocities: Velocity = Velocity + (Acceleration * Time)
    velocities = velocities + acc * dt
    
    # 3. Update positions: Position = Position + (Velocity * Time)
    positions = positions + velocities * dt

    # 4. Save a snapshot of the universe every 20 steps, or at the final step
    if step % 20 == 0 or step == num_steps - 1:
        history.append(positions.copy())

print(f"Simulation finished! Saved {len(history)} snapshots over time into our history history.")


# Ploting our Universe

Finally, we plot our results. We use Matplotlib to generate a figure containing 4 separate sub-plots.

### What the output saya
* **Initial Fluctuations (Snapshot 0)**: Particles look like a messy, random, rounded cloud. This mimics the raw matter distributions right after the big bang.
* **Early Growth (Snapshot 1)**: Gravity begins pulling nearby particles together. Tiny dense points begin to emerge.
* **Void Formation (Snapshot 2)**: As dense zones grow hungrier, they rip matter away from less dense zones. Empty spaces called "voids" form.
* **Cosmic Web (Snapshot 4)**: The final frame shows dense nodes connected by thin threads of matter ("filaments"). This is exactly how dark matter is spread throughout our real night sky!


In [ ]:
# Choose which saved snapshots from our history list to display
# Index 0 is the start, Index 1 is step 20, Index 2 is step 40, Index 4 is step 80
stages = [0, 1, 2, 4]  
titles = ['1. Initial Fluctuations', '2. Early Growth', '3. Void Formation', '4. Cosmic Web']

# Create a canvas figure that is 18 inches wide and 4.5 inches tall
# 'subplot_kw' tells Python that every single plot needs to handle 3D data depth
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), subplot_kw={'projection': '3d'})

# Loop through our 4 chosen stages and draw them one by one
for idx, stage_num in enumerate(stages):
    ax = axes[idx] # Choose the correct plot window box
    
    pos_snapshot = history[stage_num] # Grab the saved 3D positions for this era
    
    # Draw the particles as a scatter plot
    # s=4 sets particle size. alpha=0.6 makes them slightly see-through to spot clusters
    ax.scatter(pos_snapshot[:, 0], pos_snapshot[:, 1], pos_snapshot[:, 2],
               s=4, color='indigo', alpha=0.6)
    
    # Clean up the visual appearance
    ax.set_title(titles[idx], fontsize=12, fontweight='bold', pad=10)
    ax.set_xlim(-10, 10) # Lock the camera window dimensions so it doesn't auto-resize
    ax.set_ylim(-10, 10)
    ax.set_zlim(-10, 10)
    ax.axis('off')  # Turn off the gray grid box outlines for an empty space aesthetic

# Add a grand title over the top of the entire image canvas
plt.suptitle("Evolution of Dark Matter Large-Scale Structure (N-Body Simulation)", 
             fontsize=15, y=1.08, fontweight='bold', color='black')

plt.tight_layout() # Intelligently adjust spacing so nothing overlaps
plt.show() # Display the final drawing on the screen
